In [2]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

INPUT_PATH = "../data/processed/menganti_CO_preprocessed.csv"
OUTPUT_PATH = "../data/features/CO_Menganti_TSFEL.csv"

df = pd.read_csv(INPUT_PATH)

df["date"] = pd.to_datetime(df["date"])
df["CO"] = pd.to_numeric(df["CO"], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)

print("Jumlah baris :", len(df))
print("Missing CO   :", df["CO"].isna().sum())
print("Tanggal awal :", df["date"].min())
print("Tanggal akhir:", df["date"].max())

Jumlah baris : 366
Missing CO   : 0
Tanggal awal : 2025-08-31 00:00:00
Tanggal akhir: 2026-08-31 00:00:00


In [3]:
fs = 1

signal_1d = df["CO"].astype(float).values

print("Panjang signal :", len(signal_1d))
print("Tipe data      :", signal_1d.dtype)
print("Missing        :", np.isnan(signal_1d).sum())

Panjang signal : 366
Tipe data      : float64
Missing        : 0


In [4]:
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur :", len(FEATURE_LIST))

Jumlah fitur : 68


In [5]:
def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]

    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))

    return float(result)

In [6]:
def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)

    params = inspect.signature(fn).parameters

    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)

    return to_scalar(result)

In [7]:
row = {}

for fn_name in FEATURE_LIST:
    try:
        row[fn_name] = extract_one(
            fn_name,
            signal_1d,
            fs
        )
    except Exception as e:
        print(f"Gagal pada fitur {fn_name}: {e}")
        row[fn_name] = np.nan

extracted_features_final = pd.DataFrame([row])

print("Jumlah baris :", extracted_features_final.shape[0])
print("Jumlah kolom :", extracted_features_final.shape[1])
print("Missing      :", extracted_features_final.isna().sum().sum())

Jumlah baris : 1
Jumlah kolom : 68
Missing      : 0


In [8]:
print(extracted_features_final.columns.tolist())

['abs_energy', 'auc', 'autocorr', 'average_power', 'calc_centroid', 'calc_max', 'calc_mean', 'calc_median', 'calc_min', 'calc_std', 'calc_var', 'dfa', 'distance', 'ecdf', 'ecdf_percentile', 'ecdf_percentile_count', 'ecdf_slope', 'entropy', 'fundamental_frequency', 'higuchi_fractal_dimension', 'hist_mode', 'human_range_energy', 'hurst_exponent', 'interq_range', 'kurtosis', 'lempel_ziv', 'lpcc', 'max_frequency', 'max_power_spectrum', 'maximum_fractal_length', 'mean_abs_deviation', 'mean_abs_diff', 'mean_diff', 'median_abs_deviation', 'median_abs_diff', 'median_diff', 'median_frequency', 'mfcc', 'mse', 'negative_turning', 'neighbourhood_peaks', 'petrosian_fractal_dimension', 'pk_pk_distance', 'positive_turning', 'power_bandwidth', 'rms', 'skewness', 'slope', 'spectral_centroid', 'spectral_decrease', 'spectral_distance', 'spectral_entropy', 'spectral_kurtosis', 'spectral_positive_turning', 'spectral_roll_off', 'spectral_roll_on', 'spectral_skewness', 'spectral_slope', 'spectral_spread', 's

In [9]:
import os

os.makedirs("../data/features", exist_ok=True)

extracted_features_final.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"File tersimpan: {OUTPUT_PATH}")

File tersimpan: ../data/features/CO_Menganti_TSFEL.csv
